# Data Cleaning -- Testing Dataset
**This notebook prepares Flatiron Health CSV files for patients with advanced melanoma in anticipation of training a gradient-boosted survival model to predict mortality from time of first-line treatment. Specifically, it processes and cleans the test cohort.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from flatiron_cleaner import DataProcessorUrothelial
from flatiron_cleaner import merge_dataframes

## Clean CSV files 

In [2]:
test_ids = pd.read_csv('../outputs/test_patient_ids.csv')

In [3]:
test_ids = test_ids.PatientID.to_list()

In [4]:
index_date_df = pd.read_csv('../data/LineOfTherapy.csv')

In [5]:
index_date_df = (
    index_date_df
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    [['PatientID', 'StartDate']]
)

In [6]:
index_date_df.shape

(9382, 2)

In [7]:
index_date_df = (
    index_date_df[index_date_df.PatientID.isin(test_ids)]
    [['PatientID', 'StartDate']]
)

In [8]:
index_date_df.shape

(1877, 2)

In [9]:
processor = DataProcessorUrothelial()

### Process Enhanced_AdvUrothelial.csv

In [10]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvUrothelial.csv',
                                         index_date_df = index_date_df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2025-11-26 23:10:39,522 - INFO - Successfully read Enhanced_AdvUrothelial.csv file with shape: (13129, 13) and unique PatientIDs: 13129
2025-11-26 23:10:39,525 - INFO - Successfully filtered Enhanced_AdvUrothelial.csv file with shape: (1877, 14) and unique PatientIDs: 1877
2025-11-26 23:10:39,534 - INFO - Successfully processed Enhanced_AdvUrothelial.csv file with final shape: (1877, 16) and unique PatientIDs: 1877


In [11]:
enhanced_df['days_adv_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])

In [12]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [13]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [14]:
enhanced_df['PrimarySite_lower'] = enhanced_df['PrimarySite'].isin(['Bladder', 'Urethra']).astype('int64')
enhanced_df = enhanced_df.drop(columns = ['PrimarySite'])

In [15]:
# drop dates
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate',
                                          'AdvancedDiagnosisDate',
                                          'SurgeryDate',
                                          'imported_StartDate'])

### Process Demographics.csv 

In [16]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = index_date_df,
                                                 index_date_column = 'StartDate')

2025-11-26 23:10:39,560 - INFO - Successfully read Demographics.csv file with shape: (13129, 6) and unique PatientIDs: 13129
2025-11-26 23:10:39,567 - INFO - Successfully processed Demographics.csv file with final shape: (1877, 6) and unique PatientIDs: 1877


In [17]:
# Impute missing with male
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [18]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvUrothelialBiomarkers.csv

In [19]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv',
                                             index_date_df = index_date_df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2025-11-26 23:10:39,592 - INFO - Successfully read Enhanced_AdvUrothelialBiomarkers.csv file with shape: (9924, 19) and unique PatientIDs: 4251
2025-11-26 23:10:39,598 - INFO - Successfully merged Enhanced_AdvUrothelialBiomarkers.csv df with index_date_df resulting in shape: (1762, 20) and unique PatientIDs: 738
2025-11-26 23:10:39,618 - INFO - Successfully processed Enhanced_AdvUrothelialBiomarkers.csv file with final shape: (1877, 4) and unique PatientIDs: 1877


In [20]:
def map_pdl1(value):
    if pd.isna(value):  # leave missing as is
        return value
    elif value in ['0%', '< 1%']:
        return '0%'
    else:
        return '>=1%'

biomarkers_df['PDL1_binary'] = biomarkers_df['PDL1_percent_staining'].apply(map_pdl1)

In [21]:
biomarkers_df.PDL1_binary.value_counts(dropna = False)

PDL1_binary
NaN     1846
>=1%      31
Name: count, dtype: int64

In [22]:
biomarkers_df = biomarkers_df.drop(columns = ['PDL1_percent_staining'])

In [23]:
biomarkers_df['FGFR_status'] = biomarkers_df['FGFR_status'].fillna('unknown')
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

### Process ECOG.csv

In [24]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2025-11-26 23:10:39,701 - INFO - Successfully read ECOG.csv file with shape: (184794, 4) and unique PatientIDs: 9933
2025-11-26 23:10:39,728 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (32788, 5) and unique PatientIDs: 1531
2025-11-26 23:10:39,755 - INFO - Successfully processed ECOG.csv file with final shape: (1877, 3) and unique PatientIDs: 1877


In [25]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [26]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [27]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = index_date_df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-11-26 23:10:43,060 - INFO - Successfully read Vitals.csv file with shape: (3604484, 16) and unique PatientIDs: 13109
2025-11-26 23:10:44,278 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (567659, 17) and unique PatientIDs: 1877
2025-11-26 23:10:44,573 - INFO - Successfully processed Vitals.csv file with final shape: (1877, 8) and unique PatientIDs: 1877


### Process Lab.csv

In [28]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = index_date_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2025-11-26 23:10:56,733 - INFO - Successfully read Lab.csv file with shape: (9373598, 17) and unique PatientIDs: 12700
2025-11-26 23:10:58,582 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (1560658, 18) and unique PatientIDs: 1856
2025-11-26 23:11:02,044 - INFO - Successfully processed Lab.csv file with final shape: (1877, 76) and unique PatientIDs: 1877


### Process MedicationAdministration.csv

In [29]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = index_date_df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2025-11-26 23:11:03,329 - INFO - Successfully read MedicationAdministration.csv file with shape: (997836, 11) and unique PatientIDs: 10983
2025-11-26 23:11:03,578 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (156548, 12) and unique PatientIDs: 1824
2025-11-26 23:11:03,612 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (1877, 9) and unique PatientIDs: 1877


### Process Diagnosis.csv

In [30]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = index_date_df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2025-11-26 23:11:04,036 - INFO - Successfully read Diagnosis.csv file with shape: (625348, 6) and unique PatientIDs: 13129
2025-11-26 23:11:04,109 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (88030, 7) and unique PatientIDs: 1877
2025-11-26 23:11:04,393 - INFO - Successfully processed Diagnosis.csv file with final shape: (1877, 40) and unique PatientIDs: 1877


### Process Enhanced_Mortality_V2.csv

In [31]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = index_date_df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvUrothelial_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvUrothelial_Progression.csv',
                                           drop_dates = False)

2025-11-26 23:11:04,403 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (9040, 2) and unique PatientIDs: 9040
2025-11-26 23:11:04,411 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (1877, 3) and unique PatientIDs: 1877
2025-11-26 23:11:04,774 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2025-11-26 23:11:04,777 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (1877, 6) and unique PatientIDs: 1877. There are 0 out of 1877 patients with missing duration values


In [32]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [33]:
mortality_df = mortality_df.query('duration >= 0')

## Merge dataframes

In [34]:
testing_df = merge_dataframes(enhanced_df,
                              demographics_df,
                              biomarkers_df,
                              ecog_df,
                              vitals_df,
                              labs_df,
                              medications_df,
                              diagnosis_df, 
                              mortality_df,
                              merge_type = 'inner')

2025-11-26 23:11:04,797 - INFO - Anticipated number of merges: 8
2025-11-26 23:11:04,797 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 156
2025-11-26 23:11:04,798 - INFO - Dataset 1 shape: (1877, 14), unique PatientIDs: 1877
2025-11-26 23:11:04,799 - INFO - Dataset 2 shape: (1877, 6), unique PatientIDs: 1877
2025-11-26 23:11:04,799 - INFO - Dataset 3 shape: (1877, 4), unique PatientIDs: 1877
2025-11-26 23:11:04,800 - INFO - Dataset 4 shape: (1877, 4), unique PatientIDs: 1877
2025-11-26 23:11:04,800 - INFO - Dataset 5 shape: (1877, 8), unique PatientIDs: 1877
2025-11-26 23:11:04,800 - INFO - Dataset 6 shape: (1877, 76), unique PatientIDs: 1877
2025-11-26 23:11:04,801 - INFO - Dataset 7 shape: (1877, 9), unique PatientIDs: 1877
2025-11-26 23:11:04,801 - INFO - Dataset 8 shape: (1877, 40), unique PatientIDs: 1877
2025-11-26 23:11:04,802 - INFO - Dataset 9 shape: (1870, 3), unique PatientIDs: 1870
2025-11-26 23:11:04,804 - 

In [35]:
testing_df.shape

(1870, 156)

## Export dataframe

In [36]:
testing_df.to_csv('../outputs/1L_features_testing.csv', index = False)

In [37]:
# Save dtypes
testing_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/1L_features_testing_dtypes.csv')